# Learning Theory & Generalization

Companion notebook for the [Learning Theory lesson](https://ml-viz.vercel.app/courses/model-evaluation/06-learning-theory).

We make generalization concrete by fitting polynomials of growing degree: we watch the **training
vs. test error** diverge (the generalization gap), trace the classic **U-shaped** bias-variance
curve, and see the gap shrink as we add **more data**. Pure NumPy + Matplotlib.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#444', 'axes.labelcolor': '#ccc',
    'xtick.color': '#888', 'ytick.color': '#888',
    'text.color': '#eee', 'grid.color': '#333', 'lines.linewidth': 2,
})
rng = np.random.default_rng(0)

## 1 — A noisy true function

Data is a smooth function plus noise. We fit polynomials of increasing degree (increasing capacity)
and measure error on a held-out test set.

In [ ]:
def true_f(x):
    return np.sin(1.5 * x)

def make_data(n, noise=0.25, seed=0):
    r = np.random.default_rng(seed)
    x = r.uniform(-3, 3, n)
    return x, true_f(x) + r.normal(0, noise, n)

x_tr, y_tr = make_data(20, seed=1)
x_te, y_te = make_data(500, seed=2)

def fit_poly(x, y, deg):
    return np.polyfit(x, y, deg)

def mse(coef, x, y):
    return np.mean((np.polyval(coef, x) - y) ** 2)

## 2 — The generalization gap and the U-curve

As degree grows, training error keeps falling (the model fits its sample ever tighter) but test
error bottoms out and then rises — overfitting. The gap between the two curves is the
generalization gap.

In [ ]:
degrees = range(1, 16)
train_err, test_err = [], []
for d in degrees:
    c = fit_poly(x_tr, y_tr, d)
    train_err.append(mse(c, x_tr, y_tr))
    test_err.append(mse(c, x_te, y_te))

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(list(degrees), train_err, 'o-', color='#2dd4bf', label='training error')
ax.plot(list(degrees), test_err, 's-', color='#fb7185', label='test error')
best = list(degrees)[int(np.argmin(test_err))]
ax.axvline(best, ls='--', color='#888', label=f'best degree = {best}')
ax.set_yscale('log'); ax.set_xlabel('polynomial degree (capacity)'); ax.set_ylabel('MSE (log)')
ax.set_title('Training error falls forever; test error is U-shaped (bias-variance)')
ax.legend(facecolor='#1a1d27', edgecolor='#444'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print(f'best degree by test error: {best}')

## 3 — More data shrinks the gap

Fix a high-capacity model (degree 12) and grow the training set. The generalization gap
(test − train error) shrinks as n increases — formalizing 'more data generalizes better'.

In [ ]:
sizes = [12, 20, 40, 80, 160, 320]
gaps = []
for n in sizes:
    xs, ys = make_data(n, seed=7)
    c = fit_poly(xs, ys, 12)
    gaps.append(mse(c, x_te, y_te) - mse(c, xs, ys))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sizes, gaps, 'o-', color='#818cf8')
ax.set_xlabel('training set size n'); ax.set_ylabel('generalization gap (test − train MSE)')
ax.set_title('The generalization gap shrinks with more data')
ax.set_xscale('log'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
for n, g in zip(sizes, gaps):
    print(f'n={n:3d}: gap={g:.3f}')

## ✏️ Your turn

**Exercise.** Implement `generalization_gap(coef, x_tr, y_tr, x_te, y_te)` (test MSE − train MSE)
and `best_capacity(x_tr, y_tr, x_te, y_te, degrees)` returning the polynomial degree with the lowest
*test* error — the model-selection decision the U-curve is all about. Reuse `fit_poly` and `mse`.

In [ ]:
def generalization_gap(coef, x_tr, y_tr, x_te, y_te):
    # TODO(you): test MSE minus train MSE for the fitted coefficients
    return ...

def best_capacity(x_tr, y_tr, x_te, y_te, degrees):
    # TODO(you): fit each degree, return the one with the lowest test MSE
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
c = fit_poly(x_tr, y_tr, 12)
g = generalization_gap(c, x_tr, y_tr, x_te, y_te)
assert np.isclose(g, mse(c, x_te, y_te) - mse(c, x_tr, y_tr))
assert g > 0                                        # a high-capacity fit generalizes worse than it trains
bd = best_capacity(x_tr, y_tr, x_te, y_te, range(1, 16))
assert bd == best                                   # matches the U-curve minimum
# an underfit (degree 1) has higher test error than the best degree
assert mse(fit_poly(x_tr, y_tr, 1), x_te, y_te) > mse(fit_poly(x_tr, y_tr, bd), x_te, y_te)
print(f'\u2713 gap and model selection correct (best degree = {bd})')

<details>
<summary>Solution</summary>

```python
def generalization_gap(coef, x_tr, y_tr, x_te, y_te):
    return mse(coef, x_te, y_te) - mse(coef, x_tr, y_tr)

def best_capacity(x_tr, y_tr, x_te, y_te, degrees):
    return min(degrees, key=lambda d: mse(fit_poly(x_tr, y_tr, d), x_te, y_te))
```

Training error always favors the most complex model, so you must select capacity on held-out data —
the whole reason validation sets exist. The best degree balances bias (too simple) against variance
(too flexible).

</details>